# Exploring DuckDB’s Python API

This notebook contains the code examples from chapter 8 of *Getting Started with DuckDB*.

## Obtaining the dataset

To help us unpack DuckDB 's Relational API for Python, we're going to explore the *Seattle Pet Licenses* dataset, which contains information about pet licenses that have been registered in the city of Seattle. We'll be performing a small amount of **exploratory data analysis (EDA)** over this dataset, which will give us an opportunity to put DuckDB's Python client through its paces. We have included a snapshot of this dataset in this chapter's directory in the GitHub repository. This is the dataset that was used to generate the output you see in this chapter.

Alternatively, you can download the most recent version of the dataset from the Seattle Open Data portal: https://data.seattle.gov/Community/Seattle-Pet-Licenses/jguv-t9rb. You'll need to select the **CSV** option from the **Export** button. This will download the CSV file you'll need. The file that was used to produce the output shown in this chapter was **Seattle_Pet_Licenses_20240505.csv**.

## Technical requirements

In order to run the examples in this notebook, you'll need to install the Python dependencies for this project. You can do this by running the following command in your terminal when in the root directory of the project. Note that ideally this should be using a Python virtual environment for this project.

    pip install -r requirements.txt

For complete instructions on how to set up your environment for working through the examples, please consult the *Technical requirements* section of the chapter *Setting up the DuckDB Python Client*.

## Working with the Relational API

DuckDB's Relational API provides a convenient and effective Python interface for composing and working with DuckDB queries. It provides a flexible and efficient way to interact with DuckDB in Python and is especially well suited to interactive data analysis workflows.

The Relational API revolves around **DuckDBPyRe1ation** objects, which are more generally referred to as *relations*. You can think of a DuckDB relation as a representation of a DuckDB query. Relations do not contain data themselves but rather contain all the information about a query required for execution. Relations are lazily evaluated, which means that when you are working with them, nothing is run against the database until the result set for a query is needed, such as when you want to display their results interactively or when the full result set is materialized for loading into a table or exporting to a specific data format.

### Creating relations

The most general-purpose way to create a relation object is with the **DuckDBPyconnection.sq1()** method. This takes a SQL query as a string and returns a corresponding relation object representing your query. A relation object is always associated with a specific DuckDB database, which is why the **sql()** method is always called on a connection object.

In [4]:
import duckdb

pi_relation = duckdb.sql("SELECT pi() AS pi") 

type(pi_relation)

_duckdb.DuckDBPyRelation

One of the ways a relation will be executed is by displaying its results, which can be done with the **DuckDBPyRelation.show()** method.

In [2]:
pi_relation.show()

┌───────────────────┐
│        pi         │
│      double       │
├───────────────────┤
│ 3.141592653589793 │
└───────────────────┘



A convenient feature of relation objects is that their default output representation is produced by automatically calling the **show()** method. This means you can omit the **show()** method call when the expression that produces a relation object occurs at the end of a code cell in your notebook, as your IDE will automatically display its representation as output.

In [3]:
pi_relation

┌───────────────────┐
│        pi         │
│      double       │
├───────────────────┤
│ 3.141592653589793 │
└───────────────────┘

Note that the **sql()** method will only return a relation object when the SQL query contains a **SELECT** statement. Other types of SQL statements are executed immediately against the database without returning a relation since no result set needs to be returned. However, with the latest version of DuckDB, you can omit **SELECT** *.

#### Creating relations from SQL queries

In [2]:
duckdb.sql(
    """
    FROM read_csv('Seattle_Pet_Licenses.csv')
    """
)

┌────────────────────┬────────────────┬───────────────┬─────────┬──────────────────────┬─────────────────┬──────────┐
│ License Issue Date │ License Number │ Animal's Name │ Species │    Primary Breed     │ Secondary Breed │ ZIP Code │
│      varchar       │    varchar     │    varchar    │ varchar │       varchar        │     varchar     │ varchar  │
├────────────────────┼────────────────┼───────────────┼─────────┼──────────────────────┼─────────────────┼──────────┤
│ December 18 2015   │ S107948        │ Zen           │ Cat     │ Domestic Longhair    │ Mix             │ 98117    │
│ June 14 2016       │ S116503        │ Misty         │ Cat     │ Siberian             │ NULL            │ 98117    │
│ August 04 2016     │ S119301        │ Lyra          │ Cat     │ Mix                  │ NULL            │ 98121    │
│ February 13 2019   │ 962273         │ Veronica      │ Cat     │ Domestic Longhair    │ NULL            │ 98107    │
│ August 10 2019     │ S133113        │ Spider        │ 

#### Creating relations from files 

The Relational API also offers a range of methods that provide deeper integration with the Python language and that can often be more convenient methods to work with. Notably, the Relational API provides convenience methods for loading data from CSV, JSON, and Parquet files via the following methods:
- **DuckDBPYConnection.read_csv()**
- **DuckDBPyConnection.read_parquet()**
- **DuckDBPyConnection.read_json()**

Just as with the **sql()** method, these are all exposed via connection objects, meaning that they can be called against the **duckdb** module—which will use the default database—and they can be called against explicitly created connection objects. They all take a target file path for reading as a required argument and return a relation object, which is a lazily evaluated representation of the extracted contents of their target files.

In [5]:
pets_csv_relation = duckdb.read_csv("Seattle_Pet_Licenses.csv")

In [6]:
pets_csv_relation.types

[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR]

The format specifiers that can be used in the **date_format** string are documented in the DuckDB documentation: https://duckdb.org/docs/sql/functions/dateformat.html.

In [3]:
pets_csv_relation = duckdb.read_csv(
    "Seattle_Pet_Licenses.csv",
    dtype={"License Issue Date": "DATE"},
    date_format="%B %d %Y",
)

pets_csv_relation.types

[DATE, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR]

Let's quickly peek at our data. Just as you might throw a **LIMIT** clause onto a SQL query to only look a small subset of query results, we can call the **DuckDBPyRe1ation.limit()** method to do the same thing:

In [8]:
pets_csv_relation.limit(5)

┌────────────────────┬────────────────┬───────────────┬─────────┬───────────────────┬─────────────────┬──────────┐
│ License Issue Date │ License Number │ Animal's Name │ Species │   Primary Breed   │ Secondary Breed │ ZIP Code │
│        date        │    varchar     │    varchar    │ varchar │      varchar      │     varchar     │ varchar  │
├────────────────────┼────────────────┼───────────────┼─────────┼───────────────────┼─────────────────┼──────────┤
│ 2015-12-18         │ S107948        │ Zen           │ Cat     │ Domestic Longhair │ Mix             │ 98117    │
│ 2016-06-14         │ S116503        │ Misty         │ Cat     │ Siberian          │ NULL            │ 98117    │
│ 2016-08-04         │ S119301        │ Lyra          │ Cat     │ Mix               │ NULL            │ 98121    │
│ 2019-02-13         │ 962273         │ Veronica      │ Cat     │ Domestic Longhair │ NULL            │ 98107    │
│ 2019-08-10         │ S133113        │ Spider        │ Cat     │ LaPerm        

In [9]:
help(duckdb.read_csv)

Help on built-in function read_csv in module _duckdb:

read_csv(...) method of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 instance
    read_csv(path_or_buffer: object, **kwargs) -> _duckdb.DuckDBPyRelation

    Create a relation object from the CSV file in 'name'



You would do well to consult the full Python API reference documentation for all methods in the DuckDB Python API: https://duckdb.org/docs/api/python/reference.

It's worth bearing in mind that the Relational API is essentially a Python interface to the same underlying DuckDB internal interface that DuckDB SQL targets. You can, in general, use SQL queries to achieve the same results as Relational API method calls, and you can mix and match these in your workflows, depending on your needs and preferences.

In [10]:
print(pets_csv_relation.sql_query())

SELECT * FROM read_csv_auto(['Seattle_Pet_Licenses.csv'], (auto_detect = false), (delim = ','), ("quote" = '"'), (null_padding = false), ("escape" = '"'), ("comment" = chr(0)), (dateformat = '%B %d %Y'), (all_varchar = false), ("columns" = {'License Issue Date': 'DATE', 'License Number': 'VARCHAR', 'Animal's Name': 'VARCHAR', 'Species': 'VARCHAR', 'Primary Breed': 'VARCHAR', 'Secondary Breed': 'VARCHAR', 'ZIP Code': 'VARCHAR'}), (max_line_size = 2000000), (normalize_names = false), ("header" = true), ("skip" = 0), ("parallel" = true))


#### Creating relations from tables 

We can also create relations from existing tables in a DuckDB database. Since we haven't created any tables in our in-memory database yet, we'll start by making one with the **DuckDBPyRe1ation.to_table()** method. When called on its relation object, this method will create a new table with the provided name and populate it with the results of executing the relation:

In [11]:
pets_csv_relation.to_table("seattle_pets_dataset")

In [12]:
pets_table_relation = duckdb.table("seattle_pets_dataset")

In [13]:
duckdb.sql("SHOW TABLES")

┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ seattle_pets_dataset │
└──────────────────────┘

In [14]:
pets_table_relation = duckdb.table("seattle_pets_dataset")

In [15]:
pets_table_relation.describe()

┌─────────┬────────────────────┬────────────────┬────────────────────┬─────────┬───────────────────┬───────────────────┬──────────┐
│  aggr   │ License Issue Date │ License Number │   Animal's Name    │ Species │   Primary Breed   │  Secondary Breed  │ ZIP Code │
│ varchar │      varchar       │    varchar     │      varchar       │ varchar │      varchar      │      varchar      │ varchar  │
├─────────┼────────────────────┼────────────────┼────────────────────┼─────────┼───────────────────┼───────────────────┼──────────┤
│ count   │ 42567              │ 42567          │ 42526              │ 42567   │ 42567             │ 28373             │ 42440    │
│ mean    │ NULL               │ NULL           │ NULL               │ NULL    │ NULL              │ NULL              │ NULL     │
│ stddev  │ NULL               │ NULL           │ NULL               │ NULL    │ NULL              │ NULL              │ NULL     │
│ min     │ 2015-10-21         │ 015352         │ !zzy               │ Cat  

In [16]:
pets_table_relation.count("*")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        42567 │
└──────────────┘

In [17]:
duckdb.sql("DROP TABLE seattle_pets_dataset")

### Querying relations

In [4]:
pets_cleaned_relation = duckdb.sql(
    """ 
    SELECT  
        "License Issue Date" AS issue_date, 
        "Animal's Name" AS pet_name, 
        "Species" AS species, 
        "Primary Breed" AS breed 
    FROM pets_csv_relation 
    """
)

pets_cleaned_relation.limit(5)

┌────────────┬──────────┬─────────┬───────────────────┐
│ issue_date │ pet_name │ species │       breed       │
│    date    │ varchar  │ varchar │      varchar      │
├────────────┼──────────┼─────────┼───────────────────┤
│ 2015-12-18 │ Zen      │ Cat     │ Domestic Longhair │
│ 2016-06-14 │ Misty    │ Cat     │ Siberian          │
│ 2016-08-04 │ Lyra     │ Cat     │ Mix               │
│ 2019-02-13 │ Veronica │ Cat     │ Domestic Longhair │
│ 2019-08-10 │ Spider   │ Cat     │ LaPerm            │
└────────────┴──────────┴─────────┴───────────────────┘

Note that we've used a triple-quoted Python string for our query, which allows us to include new lines and also means we don't have to worry about escaping any quote characters in our SQL query.

This convenient feature of DuckDB is known as **replacement scanning**. It enables DuckDB to lookup alternative data sources when a target table name does not occur in the database catalog. In
this case, the **pets_cleaned_relation** table identifier is not present in our database's catalog. This causes DuckDB to attempt a replacement scan, identifying a variable with the same name in our Python session's global scope, which contains a **DuckDBPyRelation** object, meaning that the replacement scan will succeed and this relation with be used as the data source for this query. As we'll see later in this chapter, in addition to relation objects, replacement scans can also be used to query pandas Dataframes, Polars Dataframes, and Apache Arrow tables.

In [19]:
min_max_relation = duckdb.sql(
    """
    SELECT min(issue_date), max(issue_date)
    FROM pets_cleaned_relation
    """
)

min_max_relation

┌─────────────────┬─────────────────┐
│ min(issue_date) │ max(issue_date) │
│      date       │      date       │
├─────────────────┼─────────────────┤
│ 2015-10-21      │ 2024-04-05      │
└─────────────────┴─────────────────┘

Importantly, since DuckDB relations are lazily evaluated, we can use this technique to build up efficient data processing pipelines, enabling deferred execution until our ultimate relation object needs to be displayed or fully materialized, such as when loading into a table or exporting to a file. Chaining relation-object queries like this is analogous to SQL queries with chained references to database views; however, rather than being defined in our database's catalog, we are managing each node in the pipeline within our Python session through relation objects. It's also worth pointing out that this contrasts with using a data processing tool such as pandas to define a data pipeline within a Python session, where each intermediate result would be fully materialized, regardless of whether it is subsequentially used.

In [20]:
print(min_max_relation.explain())

In [21]:
type(min_max_relation)

_duckdb.DuckDBPyRelation

### Transformations with relations

DuckDB 's Relational API is highly flexible and can be used in several different modes to build queries that define data transformations. Here, we will go through some different modes of use to help provide you with a framework for thinking about how to use the Relational API for your own workflows.

#### Comparing SQL queries and Relational API method calling 

Broadly speaking, when you want to apply a transformation to a relation object, you can do the following:

- Define your transformations using SQL via the **sql()** method, optionally leveraging the convenience of replacement scans to query other relation objects, which allows you to compose your transformations into pipelines of multiple SQL queries
- Leverage **DuckDBPyRelation** methods that apply transformation operations and return new relation objects, enabling the composition of queries via method chaining, without the use of SQL

You can adopt either approach when querying DuckDB in Python, depending on your needs and preferences. These different styles of query creation can also be mixed and matched; you don't need to adopt one exclusively.

In [22]:
duckdb.sql(
    """
    SELECT *
    FROM pets_cleaned_relation
    WHERE species = 'Pig'
    """
)

┌────────────┬──────────┬─────────┬─────────────┐
│ issue_date │ pet_name │ species │    breed    │
│    date    │ varchar  │ varchar │   varchar   │
├────────────┼──────────┼─────────┼─────────────┤
│ 2022-11-03 │ Millie   │ Pig     │ Pot-Bellied │
│ 2023-06-01 │ Calvin   │ Pig     │ Pot-Bellied │
└────────────┴──────────┴─────────┴─────────────┘

In [23]:
pets_cleaned_relation.filter("species = 'Pig'")

┌────────────┬──────────┬─────────┬─────────────┐
│ issue_date │ pet_name │ species │    breed    │
│    date    │ varchar  │ varchar │   varchar   │
├────────────┼──────────┼─────────┼─────────────┤
│ 2022-11-03 │ Millie   │ Pig     │ Pot-Bellied │
│ 2023-06-01 │ Calvin   │ Pig     │ Pot-Bellied │
└────────────┴──────────┴─────────┴─────────────┘

Using SQL expressions with Relational API method calls enables capturing data transformations as concise Python expressions, which is particularly useful for performing rapid data analysis. Representing SQL expressions as strings does have the drawback of offering poorer integration with Python: they do not lend themselves to being composed programmatically, and they cannot support validation for subcomponents of expression, limiting opportunities for localized diagnostic feedback on errors.

#### The Expression API

DuckDB 's Python client provides an Expression API that supports the dynamic construction of Python expression objects, enabling the flexible composition of complex relation objects without the need for any SQL strings.

In [24]:
from duckdb import ColumnExpression, ConstantExpression

species_col = ColumnExpression("species")
pig_constant = ConstantExpression("Pig")

pets_cleaned_relation.filter(species_col == pig_constant)

┌────────────┬──────────┬─────────┬─────────────┐
│ issue_date │ pet_name │ species │    breed    │
│    date    │ varchar  │ varchar │   varchar   │
├────────────┼──────────┼─────────┼─────────────┤
│ 2022-11-03 │ Millie   │ Pig     │ Pot-Bellied │
│ 2023-06-01 │ Calvin   │ Pig     │ Pot-Bellied │
└────────────┴──────────┴─────────┴─────────────┘

In [25]:
pets_cleaned_relation.filter(
    "species = 'Cat' AND pet_name = 'Leeloo'"
)

┌────────────┬──────────┬─────────┬────────────────────┐
│ issue_date │ pet_name │ species │       breed        │
│    date    │ varchar  │ varchar │      varchar       │
├────────────┼──────────┼─────────┼────────────────────┤
│ 2022-04-13 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-08-15 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-08-21 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-10-20 │ Leeloo   │ Cat     │ American Shorthair │
└────────────┴──────────┴─────────┴────────────────────┘

In [26]:
name_col = ColumnExpression("pet_name")

is_cat = species_col == ConstantExpression("Cat")
is_leeloo = name_col == ConstantExpression("Leeloo")

pets_cleaned_relation.filter(is_cat & is_leeloo)

┌────────────┬──────────┬─────────┬────────────────────┐
│ issue_date │ pet_name │ species │       breed        │
│    date    │ varchar  │ varchar │      varchar       │
├────────────┼──────────┼─────────┼────────────────────┤
│ 2022-04-13 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-08-15 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-08-21 │ Leeloo   │ Cat     │ Domestic Shorthair │
│ 2022-10-20 │ Leeloo   │ Cat     │ American Shorthair │
└────────────┴──────────┴─────────┴────────────────────┘

This illustrates how the Expression API provides deeper Python integration than SQL expressions, giving us the flexibility of programmatically building up complex expressions.

#### Method chaining 

We have seen how replacement scanning enables the incremental building of queries as relation objects. Another way the Relational API enables incremental query constmction is via method chaining. Relational API methods that apply an operation to an existing relation all return a new relation that represents the updated query. The resulting relation object can have subsequent operations applied by further method calls. This allows us to build complex queries using chains of relational method calls.

In [27]:
pets_cleaned_relation.filter(is_cat).limit(5)

┌────────────┬──────────┬─────────┬───────────────────┐
│ issue_date │ pet_name │ species │       breed       │
│    date    │ varchar  │ varchar │      varchar      │
├────────────┼──────────┼─────────┼───────────────────┤
│ 2015-12-18 │ Zen      │ Cat     │ Domestic Longhair │
│ 2016-06-14 │ Misty    │ Cat     │ Siberian          │
│ 2016-08-04 │ Lyra     │ Cat     │ Mix               │
│ 2019-02-13 │ Veronica │ Cat     │ Domestic Longhair │
│ 2019-08-10 │ Spider   │ Cat     │ LaPerm            │
└────────────┴──────────┴─────────┴───────────────────┘

In [28]:
num_cat_names_rel = (
    pets_cleaned_relation
    .filter(is_cat)
    .select("pet_name")
    .distinct()
    .count("*")
)

num_cat_names_rel

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6321 │
└──────────────┘

This illustrates the power of iteratively composing queries through a series of method calls that are applied to relation objects. This is made possible by DuckDB's Relational API being designed to
support method chaining, where each method returns a new relation that incorporates the results of the operation just applied. This is reminiscent of the pandas API, in which subsequent operations can be performed on Dataframes by chaining method calls. A notable difference, however, is that internally, pandas will fully evaluate and materialize the results of each operation before invoking the next method on the results of the previous. DuckDB's Relational API is designed around lazy evaluation, deferring execution as late as possible. This means that all operations can be combined and converted into a final query before it is executed. This provides DuckDB with more opportunities for query optimization and the risk of hitting memory limitations during the calculation of intermediate operations.

In [29]:
alt_num_cat_names_rel = (
    pets_cleaned_relation
    .filter(is_cat)
    .unique("pet_name")
    .count("*")
)

alt_num_cat_names_rel

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6321 │
└──────────────┘

In [30]:
num_unique_cats_sql = alt_num_cat_names_rel.sql_query()

print(num_unique_cats_sql)

SELECT count_star() FROM (SELECT DISTINCT pet_name FROM (SELECT * FROM (WITH pets_csv_relation AS (SELECT * FROM (SELECT * FROM read_csv_auto(['Seattle_Pet_Licenses.csv'], (auto_detect = false), (delim = ','), ("quote" = '"'), (null_padding = false), ("escape" = '"'), ("comment" = chr(0)), (dateformat = '%B %d %Y'), (all_varchar = false), ("columns" = {'License Issue Date': 'DATE', 'License Number': 'VARCHAR', 'Animal's Name': 'VARCHAR', 'Species': 'VARCHAR', 'Primary Breed': 'VARCHAR', 'Secondary Breed': 'VARCHAR', 'ZIP Code': 'VARCHAR'}), (max_line_size = 2000000), (normalize_names = false), ("header" = true), ("skip" = 0), ("parallel" = true))))SELECT "License Issue Date" AS issue_date, "Animal's Name" AS pet_name, Species AS species, "Primary Breed" AS breed FROM pets_csv_relation) AS unnamed_relation_555aa29db4db5b54 WHERE (species = 'Cat')) AS unnamed_relation_555aa29db4db5b54) AS unnamed_relation_555aa29db4db5b54 GROUP BY ALL


In [31]:
import sqlparse

formatted_sql = sqlparse.format(
    num_unique_cats_sql,
    reindent=True
)
print(formatted_sql)

SELECT count_star()
FROM (
SELECT DISTINCT pet_name
FROM (
SELECT *
FROM (WITH pets_csv_relation AS (
SELECT *
FROM
  (SELECT *
   FROM read_csv_auto(['Seattle_Pet_Licenses.csv'], (auto_detect = false), (delim = ','), ("quote" = '"'), (null_padding = false), ("escape" = '"'), ("comment" = chr(0)), (dateformat = '%B %d %Y'), (all_varchar = false), ("columns" = {'License Issue Date': 'DATE', 'License Number': 'VARCHAR', 'Animal's Name': 'VARCHAR', 'Species': 'VARCHAR', 'Primary Breed': 'VARCHAR', 'Secondary Breed': 'VARCHAR', 'ZIP Code': 'VARCHAR'}), (max_line_size = 2000000), (normalize_names = false), ("header" = true), ("skip" = 0), ("parallel" = true))))SELECT "License Issue Date" AS issue_date, "Animal's Name" AS pet_name, Species AS species, "Primary Breed" AS breed
                                                                                                                                                                                                                           

#### Putting it all together

In [32]:
duckdb.sql(
    """
    SELECT *, length(pet_name) AS name_length 
    FROM pets_cleaned_relation 
    ORDER BY name_length DESC
    LIMIT 10
    """
)

┌────────────┬────────────────────────────────────────────────────┬─────────┬────────────────────────────┬─────────────┐
│ issue_date │                      pet_name                      │ species │           breed            │ name_length │
│    date    │                      varchar                       │ varchar │          varchar           │    int64    │
├────────────┼────────────────────────────────────────────────────┼─────────┼────────────────────────────┼─────────────┤
│ 2023-08-04 │ Nuit Ahathoor Hecate Sappho Jezebel Lilith Crowley │ Dog     │ Chihuahua, Short Coat      │          50 │
│ 2024-02-10 │ Lady Kassandra Yu Countess of Wallingford DBE      │ Dog     │ Border Collie              │          45 │
│ 2024-02-01 │ KingKing SirBeastmodeEsquire Stella Jr Sr II       │ Dog     │ Terrier, American Pit Bull │          44 │
│ 2024-01-09 │ Alyeska Juniper Cocoa Luna Taber O'Kelley          │ Cat     │ Domestic Shorthair         │          41 │
│ 2024-01-22 │ WINTERDAWG PRINCE

In [6]:
from duckdb import (
    ColumnExpression,
    FunctionExpression,
    StarExpression
)

star = StarExpression()
name_col = ColumnExpression("pet_name")
name_length_col = (
    FunctionExpression("length", name_col)
    .alias("name_length")
)
name_length_sort = ColumnExpression("name_length").desc()

longest_names_relation = (
    pets_cleaned_relation
    .select(star, name_length_col)
    .sort(name_length_sort)
    .limit(10)
)

longest_names_relation

┌────────────┬────────────────────────────────────────────────────┬─────────┬────────────────────────────┬─────────────┐
│ issue_date │                      pet_name                      │ species │           breed            │ name_length │
│    date    │                      varchar                       │ varchar │          varchar           │    int64    │
├────────────┼────────────────────────────────────────────────────┼─────────┼────────────────────────────┼─────────────┤
│ 2023-08-04 │ Nuit Ahathoor Hecate Sappho Jezebel Lilith Crowley │ Dog     │ Chihuahua, Short Coat      │          50 │
│ 2024-02-10 │ Lady Kassandra Yu Countess of Wallingford DBE      │ Dog     │ Border Collie              │          45 │
│ 2024-02-01 │ KingKing SirBeastmodeEsquire Stella Jr Sr II       │ Dog     │ Terrier, American Pit Bull │          44 │
│ 2024-01-09 │ Alyeska Juniper Cocoa Luna Taber O'Kelley          │ Cat     │ Domestic Shorthair         │          41 │
│ 2024-01-22 │ WINTERDAWG PRINCE

In [21]:
longest_names_sql = longest_names_relation.sql_query()

print(longest_names_sql)

SELECT * FROM (SELECT *, length(pet_name) AS name_length FROM (WITH pets_csv_relation AS (SELECT * FROM (SELECT * FROM read_csv_auto(['Seattle_Pet_Licenses.csv'], (auto_detect = false), (delim = ','), ("quote" = '"'), (null_padding = false), ("escape" = '"'), ("comment" = chr(0)), (dateformat = '%B %d %Y'), (all_varchar = false), ("columns" = {'License Issue Date': 'DATE', 'License Number': 'VARCHAR', 'Animal's Name': 'VARCHAR', 'Species': 'VARCHAR', 'Primary Breed': 'VARCHAR', 'Secondary Breed': 'VARCHAR', 'ZIP Code': 'VARCHAR'}), (max_line_size = 2000000), (normalize_names = false), ("header" = true), ("skip" = 0), ("parallel" = true))))SELECT "License Issue Date" AS issue_date, "Animal's Name" AS pet_name, Species AS species, "Primary Breed" AS breed FROM pets_csv_relation) AS unnamed_relation_78d52b014ea84651) AS unnamed_relation_78d52b014ea84651 ORDER BY name_length DESC LIMIT 10


In [22]:
longest_names_relation.explain()

''

### Adopting a querying approach

One of the primary benefits of leaning into the Relational API's dataframe-like mode of use is that it gives you tighter integration with Python language features, allowing you to interact with query components as first-class Python objects and dynamically construct them programmatically. This is particularly useful when developing DuckDB-powered products and services, where query logic may need to be customized at mntime in response to user input and other dynamic parameters. Another benefit is that, in contrast to defining monolithic queries as SQL strings, iteratively composing relations and creating expression objects will tend to provide more localized—and, therefore, more useful—query validation errors, providing a better development experience.

### Writing to disk 

In [ ]:
pets_cleaned_relation.write_csv("seattle_pets.csv")

pets_cleaned_relation.write_parquet("seattle_pets.parquet")

In [8]:
duckdb.sql(
    "COPY pets_cleaned_relation TO 'seattle_pets.csv'"
)

duckdb.sql(
    "COPY pets_cleaned_relation TO 'seattle_pets.parquet'"
)

In [9]:
duckdb.sql(
    "COPY pets_cleaned_relation TO 'seattle_pets.json'"
)

You can find the assorted parameters that can be used to configure exporting to different file formats in the DuckDB documentation for the COPY statement: https://duckdb.org/docs/sql/statements/copy.html.

### Modifying the database

We can also use the Relational API to make modifications to databases.

In [10]:
conn = duckdb.connect("seattle_pets.db")

In [ ]:
conn.sql("DROP TABLE IF EXISTS pets")

In [14]:
conn.read_parquet("seattle_pets.parquet").to_table("pets")

In [15]:
conn.sql("SHOW TABLES")

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ pets    │
└─────────┘

In [16]:
conn.table("pets").count("*")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        42567 │
└──────────────┘

In [17]:
new_dog1 = ("2024-05-19", "Monty", "Dog", "Border Collie") 

# insert expects a list of objects, so convert the tuple to a list
conn.table("pets").insert(list(new_dog1)) 

In [18]:
new_dog2 = ("2024-05-19", "Pixie", "Dog", "Australian Kelpie") 

new_dog_rel = conn.values(list(new_dog2)) 

new_dog_rel.insert_into("pets") 

In [19]:
conn.table("pets").filter("issue_date = '2024-05-19'")

┌────────────┬──────────┬─────────┬───────────────────┐
│ issue_date │ pet_name │ species │       breed       │
│    date    │ varchar  │ varchar │      varchar      │
├────────────┼──────────┼─────────┼───────────────────┤
│ 2024-05-19 │ Monty    │ Dog     │ Border Collie     │
│ 2024-05-19 │ Pixie    │ Dog     │ Australian Kelpie │
└────────────┴──────────┴─────────┴───────────────────┘

In [20]:
conn.close()

For further reference, we suggest consulting the DuckDB documentation on the Relational API: https://duckdb.org/docs/api/python/relational_api.

## Working with the Python DB-API

The DuckDB Python client also provides another API for interacting with DuckDB databases: the Python DB-API. This is an interface that is compliant with the **Python Database API Specification v2.0 (DB-API 2.0)**, which is described by PEP 249 (https://peps.python.org/pep-0249). The DB-API specification describes a standardized interface for accessing databases within Python, which has become a commonly used standard for Python database client packages. By encouraging conformity of API interfaces across different Python database libraries, Python code that targets this interface is more readily understood and more portable across databases, leading to a general increase in the availability of database connectivity in the Python ecosystem.

Note that this is not intended to be an exhaustive treatment of DuckDB's DB-API. For that, we encourage you to consult the DuckDB documentation: https://duckdb.org/docs/api/python/dbapi.

### Connecting to a database 

In [7]:
conn = duckdb.connect()

### Querying databases

In [9]:
conn.execute(
    """
    CREATE OR REPLACE TABLE seattle_pets AS
    SELECT * FROM 'seattle_pets.parquet'
    """
)

In [10]:
conn.execute("SELECT * FROM seattle_pets") 

In [11]:
conn.fetchone()

(datetime.date(2015, 12, 18), 'Zen', 'Cat', 'Domestic Longhair')

In [12]:
conn.description

[('issue_date', DATE, None, None, None, None, None),
 ('pet_name', VARCHAR, None, None, None, None, None),
 ('species', VARCHAR, None, None, None, None, None),
 ('breed', VARCHAR, None, None, None, None, None)]

In [13]:
[conn.fetchone() for i in range(3)]

[(datetime.date(2016, 6, 14), 'Misty', 'Cat', 'Siberian'),
 (datetime.date(2016, 8, 4), 'Lyra', 'Cat', 'Mix'),
 (datetime.date(2019, 2, 13), 'Veronica', 'Cat', 'Domestic Longhair')]

In [14]:
conn.fetchmany(3)

[(datetime.date(2019, 8, 10), 'Spider', 'Cat', 'LaPerm'),
 (datetime.date(2019, 11, 21), 'Maxx', 'Cat', 'American Shorthair'),
 (datetime.date(2020, 5, 24), 'Mickey', 'Cat', 'Domestic Longhair')]

In [15]:
rest_rows = conn.fetchall()

len(rest_rows)

42560

### Running SQL queries using Prepared statements

In [16]:
import datetime

new_pet1 = [
    datetime.date.today(),
    "Ned",
    "Dog",
    "Border Collie"
]

conn.execute(
    "INSERT INTO seattle_pets VALUES (?, ?, ?, ?)", 
    parameters=new_pet1
)

In [56]:
new_pet2 = {
    "name": "Simon",
    "species": "Cat",
    "breed": "Bombay",
    "issue_date": datetime.date.today(),
}

conn.execute(
    """
    INSERT INTO seattle_pets VALUES
        ($issue_date, $name, $species, $breed)
    """,
    new_pet2
)

In [57]:
conn.execute(
    """ 
    SELECT * 
    FROM seattle_pets 
    WHERE issue_date = ?; 
    """,
    [datetime.date.today()],
).fetchall()

[(datetime.date(2025, 10, 21), 'Ned', 'Dog', 'Border Collie'),
 (datetime.date(2025, 10, 21), 'Simon', 'Cat', 'Bombay')]

### Writing to disk

In [58]:
conn.execute(
    "COPY seattle_pets TO 'seattle_pets_updates.csv'"
) 

conn.execute(
    "COPY seattle_pets TO 'seattle_pets_updates.parquet'"
)

### Closing the database connection

In [59]:
conn.close()

### Database cursors

In [60]:
conn = duckdb.connect()

new_conn = conn.cursor()

## Integration with Python packages and language features 

### Querying Python data structures

#### Querying Python objects via replacement scans 

In [5]:
import pandas as pd  

pets_df = pd.read_parquet("seattle_pets.parquet")

duckdb.sql("SELECT * FROM pets_df USING SAMPLE 1")

┌────────────┬──────────┬─────────┬────────────────────┐
│ issue_date │ pet_name │ species │       breed        │
│    date    │ varchar  │ varchar │      varchar       │
├────────────┼──────────┼─────────┼────────────────────┤
│ 2023-02-05 │ Poonai   │ Cat     │ Domestic Shorthair │
└────────────┴──────────┴─────────┴────────────────────┘

#### Registering objects as DuckDB views 

In [4]:
pets_dict = {
    "seattle": pd.read_parquet("seattle_pets.parquet")
}

duckdb.register("pets_view", pets_dict["seattle"])

duckdb.sql("SELECT * FROM pets_view USING SAMPLE 1")

┌────────────┬──────────┬─────────┬────────────────────┐
│ issue_date │ pet_name │ species │       breed        │
│    date    │ varchar  │ varchar │      varchar       │
├────────────┼──────────┼─────────┼────────────────────┤
│ 2022-12-14 │ Fiona    │ Cat     │ Domestic Shorthair │
└────────────┴──────────┴─────────┴────────────────────┘

#### Creating tables from objects 

In [6]:
pets_df = pd.read_parquet("seattle_pets.parquet")

duckdb.sql(
    """
    CREATE OR REPLACE TABLE pets_table_from_df AS
    SELECT * FROM pets_df
    """
)

duckdb.sql("SELECT * FROM pets_table_from_df USING SAMPLE 1")

┌────────────┬──────────┬─────────┬────────────────────┐
│ issue_date │ pet_name │ species │       breed        │
│    date    │ varchar  │ varchar │      varchar       │
├────────────┼──────────┼─────────┼────────────────────┤
│ 2023-01-22 │ Gus      │ Cat     │ Domestic Shorthair │
└────────────┴──────────┴─────────┴────────────────────┘

### Converting query results

We have already seen the **fetchone(), fetchmany()**, and **fetchall()** methods that retrieve records as either a singel tuple or a list of tuples. Let's look at some of the other available data formats. These are available against both relation objects and connection objects, which means that youc an use the following methods regardless of whether you're using the Relational API or the DB-API:
- **df()**: Retrieves results as a pandas dataframe (pandas must be installed)
- **pl()**: Retrieves results as a Polars dataframe (Polars must be installed)
- **arrow()**: Retrieves results as an Arrow table (PyArrow must be installed)
- **fetchnumpy()**: Retrieves results as a dictionary of NumPy arrays (NumPy must be installed)
- **torch()**: Retrieves results as a dictionary of PyTorch Tensors (PyTorch must be installed)
- **tf()**: Retrieves results as a dictionary of TensorFlow Tensors (TensorFlow must be installed)

Note that this is not an exhaustive list; see the DuckDB documentation for reference (https://duckdb.org/docs/api/python/result_conversion).

#### Converting to dataframes 

A common workflow when using DuckDB is analytical workflows invovles qauerying a DuckDB database and then converting the results to a dataframe such as pandas or Polars, continuing to perform your analysis with the dataframe. This combines the strengths of both DuckDB and dataframe libraries, allowing you to crunch analytical queries over large datasets on your local machine than most dataframe tools are able to handle, and then converting smaller result sets into a now manageable dataframe whose characteristic features lend themselves to effectively preparing and analyzing data.

Right now, let's look at a very simple example that involves loading our Seattle pets Parquet file intoa DuckDB relation and converting its results into a pandas dataframe:

In [8]:
conn = duckdb.connect()
seattle_pets = conn.from_parquet("seattle_pets.parquet")

pandas_df = seattle_pets.df()
pandas_df[
    pandas_df["species"] == "Dog"
].value_counts("breed")[:5]

breed
Retriever, Labrador      3034
Retriever, Golden        1488
Chihuahua, Short Coat    1443
German Shepherd           954
Poodle, Miniature         846
Name: count, dtype: int64

Let's now illustration the conversion to a Polars dataframe by using it to analyze the top five cat breeds:

In [9]:
import polars as pl

polars_df = seattle_pets.pl()
polars_df.filter(
    pl.col("species") == "Cat"
)["breed"].value_counts(sort=True)[:5]

breed,count
str,u32
"""Domestic Shorthair""",7408
"""Domestic Medium Hair""",1555
"""American Shorthair""",1301
"""Domestic Longhair""",881
"""Siamese""",457


#### Converting to Arrow tables

Apache Arrow is a particularly useful in-memory data format that DuckDB can convert results into. A notable benefit in converting to Arrow tables is that this will be a zero-copy operation from the
DuckDB database, which means the desired data does not need to be copied to a new memory location for the new data structure, saving both memory bandwidth and CPU cycles. Converting to
other data formats will involve some degree of conversion overheads, which you may need to consider for certain applications.

Arrow tables are also useful as a data interchange format, as many popular in-memory data format libraries can consume Arrow tables. This makes it useful for converting to other data formats, such as Vaex and Apache Arrow DataFusion, which DuckDB does not support converting to directly at the time of writing.

In [10]:
conn = duckdb.connect()

conn.execute("SELECT * FROM 'seattle_pets.parquet'")

pets_table = conn.arrow()

pets_table.schema

issue_date: date32[day]
pet_name: string
species: string
breed: string

### Data types: from Python to DuckDB

DuckDB has a comprehensive range of data types, which are outlined in the DuckDB documentation: https://duckdb.org/docs/sql/data_types/overview.

When working with the Python DuckDB client, one consideration is how Python types are mapped to DuckDB types. Within the Python API, DuckDB types are represented by instances of the **DuckDBPyType** class. You can access these instances directly via the **duckdb.type** module:

In [1]:
# duckdb.typing is not available in this environment; use SQL type names instead
# varchar_type = duckdb.typing.VARCHAR
# bigint_type = duckdb.typing.BIGINT
varchar_type = "VARCHAR"
bigint_type = "BIGINT"

In general, any method in the DuckDB Python API that accepts a **DuckDBPyType** instance can be a Python type, and it will be automatically converted according to the mapping found in the DuckDB Python client documentation: https://duckdb.org/docs/api/python/types. You can also manually create instances of **DuckDBPyType** by passing the desired Python type, which will be mapped to the corresponding DuckDB data type:

In [ ]:
# duckdb.typing is not available in this environment; use SQL type names instead
# (these match the values used earlier in the notebook)
# varchar_type = duckdb.typing.DuckDBPyType(str)
# bigint_type = duckdb.typing.DuckDBPyType(int)
varchar_type = "VARCHAR"
bigint_type = "BIGINT"

Python values can also be coerced into appropriate DuckDB values where possible. To see this in action, we can use the **values()** method from the Relational API, which takes a sequence of values and returns a DuckDB relation representing these values as a single row with appropriately typed columns.

In [17]:
duckdb.values(
    [
        10,
        1_000_000,
        0.95,
        "hello string",
        b"hello bytes",
        True,
        datetime.date.today(),
        None,
    ]
)

┌───────┬─────────┬────────┬──────────────┬─────────────┬─────────┬────────────┬───────┐
│ col0  │  col1   │  col2  │     col3     │    col4     │  col5   │    col6    │ col7  │
│ int32 │  int32  │ double │   varchar    │    blob     │ boolean │    date    │ int32 │
├───────┼─────────┼────────┼──────────────┼─────────────┼─────────┼────────────┼───────┤
│    10 │ 1000000 │   0.95 │ hello string │ hello bytes │ true    │ 2025-11-15 │  NULL │
└───────┴─────────┴────────┴──────────────┴─────────────┴─────────┴────────────┴───────┘

Also supported is the conversion of nested data types, such as lists, tuples, and dictionaries:

In [19]:
duckdb.values(
    [
        (1, 2), 
        ["hello", "world"],
        {"key1": 10, "key2": "quack!"}
    ]
)

┌─────────┬────────────────┬────────────────────────────────────┐
│  col0   │      col1      │                col2                │
│ int32[] │   varchar[]    │ struct(key1 integer, key2 varchar) │
├─────────┼────────────────┼────────────────────────────────────┤
│ [1, 2]  │ [hello, world] │ {'key1': 10, 'key2': quack!}       │
└─────────┴────────────────┴────────────────────────────────────┘

Both the tuple and the list values have been converted to DuckDB **LIST** types, which are ordered sequences all of the same type. We can also see that the dictionary has been converted into a DuckDB **STRUCT** type; structs are mappings between string keys and values, whose type can vary across each key.
This has provided a quick tour of how to think about the interface between Python and DuckDB types. For more details, see the DuckDB documentation for working with Python types (https://
duckdb.org/docs/api/python/types) and the DuckDB data types reference (https://duckdb.org/docs/sql/data_types/overview).

### User-defined functions

There are times when you might find yourself wanting some capabilities provided by Python or its ecosystem of packages when writing DuckDB SQL queries. For such situations, DuckDB offers the ability to create user-defined functions (UDFs). This enables you to register Python-defined functions within a DuckDB database and use them as though they were DuckDB SQL functions.

Let's imagine that we wanted to augment our pets dataset with a column containing an emoji corresponding to the species of each pet's registration record. We'll start by defining a Python function, **emojify()**, that makes use of the Python emoji package (https://github.com/carpedm20/emoji) to convert a species name into a string containing the corresponding emoji. Note that the **emoji** package will have been installed by running **pip install emoji** on the command line.

In [20]:
import emoji

def emojify(species):
    """Converts a string into a single emoji."""
    emoji_str = emoji.emojize(f":{species.lower()}:")
    if emoji.is_emoji(emoji_str):
        return emoji_str
    return None

In [36]:
emojify("goat")

'🐐'

We get a string containing a goat emoji as our output. We want to be able to apply this to every pet record, so the next thing we need to do is register this function with DuckDB. To do this, we need to use the create_function() method. In this case, we're registering it on the default database, but we could also be calling it against a specific connection object. To register a function, we need to specify the following information as arguments to the create_function() method:
- The function name will be registerd as in DuckDB; we'll name it the same as the Python function name.
- The Python function object to register.
- A list of DuckDB types corresponding to the type of each function argument. In this case, it will be a single **VARCHAR** type since our Python function takes a single argument of type **str**.
- The return type of the function, which will also be **VARCHAR**.

In [ ]:
# Use SQL type names as strings because duckdb.typing is not available in this environment
duckdb.create_function(
    "emojify",
    emojify,
    ["VARCHAR"], #[duckdb.typing.VARCHAR],
    "VARCHAR", #[duckdb.typing.VARCHAR]
)

With the emoji function registered, we can now use it in a SQL query as if it were a DuckDB SQL function. Let's query the Parquet file we saved with updated additional pet records, returning all columns as well as the newly derived emoji column that contains the emoji representation of each pet.

In [76]:
duckdb.sql(
    """ 
    SELECT *, emojify(species) AS emoji 
    FROM 'seattle_pets_updates.parquet' 
    USING SAMPLE 10 
    """
)

┌────────────┬─────────────┬─────────┬─────────────────────────────────────┬─────────┐
│ issue_date │  pet_name   │ species │                breed                │  emoji  │
│    date    │   varchar   │ varchar │               varchar               │ varchar │
├────────────┼─────────────┼─────────┼─────────────────────────────────────┼─────────┤
│ 2022-08-09 │ Sara        │ Cat     │ Mix                                 │ 🐈      │
│ 2022-11-22 │ Trillium    │ Cat     │ Domestic Shorthair                  │ 🐈      │
│ 2023-12-11 │ Lucy        │ Cat     │ American Shorthair                  │ 🐈      │
│ 2023-11-11 │ Dolly       │ Dog     │ Retriever, Golden                   │ 🐕      │
│ 2022-05-05 │ Mr. Pickles │ Dog     │ Schnauzer, Miniature                │ 🐕      │
│ 2022-09-16 │ Pepper Jack │ Dog     │ Retriever, Labrador                 │ 🐕      │
│ 2023-02-24 │ Roger       │ Dog     │ Poodle, Standard                    │ 🐕      │
│ 2023-06-30 │ Lewis       │ Dog     │ Retriever, 

In order to remove the UDF from the database, we just call the **remove_function()** method:

In [39]:
duckdb.remove_function("emojify")

In the previous section on Python and DuckDB typing, we mentioned that anywhere a DuckDB
type is required in a DuckDB Python API method, the corresponding Python type can be used instead, and it will be automatically converted. Let's see that in action with the **create_function()** method:

In [40]:
duckdb.create_function("emojify", emojify, [str], str)

We can simplify the registration of this function further by defining our Python **emojify()** function using type annotations on both the function arguments and the return value. DuckDB can inspect these annotations and automatically infer the correct types for registering the function:

In [41]:
def emojify(species: str) -> str:
    """Converts a string into a single emoji."""
    emoji_str = emoji.emojize(f":{species.lower()}:")
    if emoji.is_emoji(emoji_str):
        return emoji_str
    return None

duckdb.remove_function("emojify")

duckdb.create_function("emojify", emojify)

#### Performance Considerations for Python UDFs

An impotfant consideration around the use of Python IJDFs is that they will introduce a performance hit when compared with using native DuckDB functions. When a UDF is called in a query, DuckDB will process chunks of data individually by threading each one through the local Python interpreter, which will slow your query down. DuckDB does support the creation of more efficient vectorized UDFs that accept Arrow arrays as input; however, there will still be a performance penalty incurred. Where you have the option, you should therefore use native DuckDB functions, reaching for IJDFs only when DuckDB doesn't provide your required functionality.

DuckDB's UDF support offers a few more features that we won't go into here, such as creating more efficient vectorized UDFs that accept Arrow arrays as input, enabling functions that have side effects, and controlling how the function behaves when NULL values are passed as inputs. We refer you to the DuckDB documentation for a more complete treatment of its IJDF support: https://duckdb.org/docs/api/python/function.

### Handling exceptions

When writing application code, it's good practice to make your code resilient to runtime exceptions through the use of error handling. The DuckDB Python client has a collection of exception classes that it uses for raising appropriate exceptions given the error that has occurred. Following exception-handling best practice, you should, in general, aim to catch the most specific exception type that could arise within the line or lines of code you are guarding. The DuckDB exception classes are located at the top level of the duckdb module.

Let's start with how you can handle errors that arise when DuckDB cannot convert a value to a target type:

In [80]:
from duckdb import ConversionException

try:
    duckdb.execute("SELECT '5,000'::INTEGER").fetchall()
except ConversionException as error:
    print(error)
    # handle exception...

Conversion Error: Could not convert string '5,000' to INT32

LINE 1: SELECT '5,000'::INTEGER
                      ^


The preceding snippet catches the DuckDB conversionException exception, which was thrown
due to a conversion from string to integer failing on account of a non-digit character occurring in the string. For the next type of error, let's look at how you can guard against DuckDB making a query against a non-existent table in the database catalog:

In [42]:
from duckdb import CatalogException

try:
    duckdb.sql("SELECT * from imaginary_table")
except CatalogException as error:
    print(error)
    # handle exception...

Catalog Error: Table with name imaginary_table does not exist!
Did you mean "pragma_database_list"?


For a complete list of available exceptions provided by the DuckDB Python client, see the
full DuckDB Python client API reference: https://duckdb.org/docs/api/python/reference.

## Summary

This chapter was a deep dive through the DuckDB Python client. We started with the Relational API, illustrating its affinity for analytical workflows by using it to explore the Seattle pet licenses dataset. We then took the DB-API through its paces, illustrating some of the operations that you can use to build data applications and integrations. We then finished the chapter with a look at a range of Python integrations that DuckDB offers, including working with other data structures, converting results into other formats, type conversion between Python and DuckDB, creating UDFs, and lastly, handling exceptions raised by DuckDB. Do note that this chapter and the previous one are not intended to be an exhaustive guide to the DuckDB Python API. For that, you should
consult the documentation for the DuckDB Python API: https://duckdb.org/docs/api/python.

You now have the ingredients you need to bring DuckDB into your Python data analysis
workflows or to start building data applications or integrations in Python using DuckDB. After going through the Python DuckDB client in some depth, in the next chapter, we will jump into using the DuckDB client for R, a language designed specifically to support statistical computing applications and that is notable for its popularity among scientists, data scientists, and data analysts.